In [1]:
pip install gdown

In [2]:
# Download the zip file from Google Drive
import gdown
import os

file_id = '1IHoMOnmA_TUGwBoUlQIAwb8nokqs5at8'
output_filename = 'archive.zip'

gdown.download(id=file_id, output=output_filename, quiet=False)

Downloading...
From (original): https://drive.google.com/uc?id=1IHoMOnmA_TUGwBoUlQIAwb8nokqs5at8
From (redirected): https://drive.google.com/uc?id=1IHoMOnmA_TUGwBoUlQIAwb8nokqs5at8&confirm=t&uuid=eb53caf9-9b13-4ff6-85a4-3169f31efcff
To: /content/archive.zip
100%|██████████| 1.27G/1.27G [00:09<00:00, 139MB/s]


'archive.zip'

In [3]:
# Extract the contents of the zip file
import zipfile

if os.path.exists(output_filename):
    with zipfile.ZipFile(output_filename, 'r') as zip_ref:
        zip_ref.extractall('.') # Extract to the current directory
    print(f"Successfully extracted '{output_filename}'")
    # Optionally, remove the zip file after extraction
    # os.remove(output_filename)
else:
    print(f"Error: The file '{output_filename}' was not found.")

Successfully extracted 'archive.zip'


In [4]:
# List the contents of the current directory to see extracted files
print("Contents of the current directory after extraction:")
print(os.listdir('.'))

Contents of the current directory after extraction:
['.config', 'archive.zip', 'dataset', 'sample_data']


In [5]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

In [6]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split

transform = transforms.Compose([
    transforms.Resize((110, 100)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
])

full_dataset = datasets.ImageFolder(root='dataset', transform=transform)

In [7]:
train_size = int(0.7 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

In [8]:
# changed batch size
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=2, shuffle=False)


In [9]:
num_classes = len(full_dataset.classes)
print(f"Number of classes in the dataset: {num_classes}")

Number of classes in the dataset: 4


In [10]:
from collections import Counter

# Count classes in train_dataset
train_class_counts = Counter()
for _, labels in train_loader:
    train_class_counts.update(labels.cpu().numpy())

print("Number of images per class in Training Set:")
for class_idx, count in sorted(train_class_counts.items()):
    class_name = full_dataset.classes[class_idx]
    print(f"  Class {class_name} (ID: {class_idx}): {count} images")

# Count classes in val_dataset
val_class_counts = Counter()
for _, labels in val_loader:
    val_class_counts.update(labels.cpu().numpy())

print("\nNumber of images per class in Validation Set:")
for class_idx, count in sorted(val_class_counts.items()):
    class_name = full_dataset.classes[class_idx]
    print(f"  Class {class_name} (ID: {class_idx}): {count} images")

Number of images per class in Training Set:
  Class broadleaf (ID: 0): 830 images
  Class grass (ID: 1): 2447 images
  Class soil (ID: 2): 2292 images
  Class soybean (ID: 3): 5166 images

Number of images per class in Validation Set:
  Class broadleaf (ID: 0): 361 images
  Class grass (ID: 1): 1073 images
  Class soil (ID: 2): 957 images
  Class soybean (ID: 3): 2210 images


MODEL Custom Model1

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomCNN(nn.Module):
    def __init__(self, num_classes=4):
        super(CustomCNN, self).__init__()

        self.conv1 = nn.Conv2d(3, 32, 3)
        self.conv2 = nn.Conv2d(32, 16, 3)
        self.conv3 = nn.Conv2d(16, 8, 3)
        self.conv4 = nn.Conv2d(8, 4, 3)

        self.pool = nn.MaxPool2d(2, 2)

        # ❗ IMPORTANT: 80 input features
        self.fc1 = nn.Linear(80, 128)
        self.fc2 = nn.Linear(128, 128)
        self.fc3 = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = self.pool(F.relu(self.conv4(x)))  # KEEP this

        x = x.view(x.size(0), -1)  # → 80

        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x)

        return x

In [12]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = CustomCNN(num_classes=4).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

training loop

In [13]:
def train_model(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # Validation
        model.eval()
        val_loss = 0
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)

                loss = criterion(outputs, labels)
                val_loss += loss.item()

                _, predicted = torch.max(outputs, 1)
                total += labels.size(0)
                correct += (predicted == labels).sum().item()

        print(f"Epoch [{epoch+1}/{epochs}]")
        print(f"Train Loss: {train_loss/len(train_loader):.4f}")
        print(f"Val Loss: {val_loss/len(val_loader):.4f}")
        print(f"Val Accuracy: {100 * correct / total:.2f}%\n")

In [14]:
train_model(model, train_loader, val_loader, epochs=10)

Epoch [1/10]
Train Loss: 0.5566
Val Loss: 0.3535
Val Accuracy: 86.48%

Epoch [2/10]
Train Loss: 0.3656
Val Loss: 0.3488
Val Accuracy: 86.63%

Epoch [3/10]
Train Loss: 0.3306
Val Loss: 0.2918
Val Accuracy: 87.85%

Epoch [4/10]
Train Loss: 0.3036
Val Loss: 0.2692
Val Accuracy: 88.92%

Epoch [5/10]
Train Loss: 0.2890
Val Loss: 0.2527
Val Accuracy: 89.31%

Epoch [6/10]
Train Loss: 0.2563
Val Loss: 0.2198
Val Accuracy: 91.22%

Epoch [7/10]
Train Loss: 0.2308
Val Loss: 0.2147
Val Accuracy: 91.63%

Epoch [8/10]
Train Loss: 0.2221
Val Loss: 0.2994
Val Accuracy: 88.26%

Epoch [9/10]
Train Loss: 0.2130
Val Loss: 0.1710
Val Accuracy: 93.00%

Epoch [10/10]
Train Loss: 0.1983
Val Loss: 0.1877
Val Accuracy: 92.68%



In [15]:
total_params = sum(p.numel() for p in model.parameters())
print("Total parameters:", total_params)

Total parameters: 34368
